# Vehicle CO₂ Emissions Prediction

**Project type:** Supervised machine learning · Regression  
**Target:** `CO2 Emissions(g/km)`  
**Dataset:** Canadian vehicle fuel consumption and CO₂ emissions data

## Objective
Train and compare regression models to estimate a vehicle's CO₂ emissions (g/km) from its technical and fuel-consumption characteristics.

## Models
- Linear Regression
- Random Forest Regressor (selected numerical features)
- Random Forest Regressor (all available features, with categorical encoding)
- XGBoost Regressor

## Repository setup
Place `CO2 Emissions_Canada.csv` in the same directory as this notebook, then run the cells from top to bottom. The dataset is not embedded in this notebook; check its source and license before redistributing it.

**Note:** Evaluation metrics describe performance on the held-out test split. They do not establish that the model will generalize to other vehicle populations or future model years.


In [ ]:
from pathlib import Path

DATA_PATH = Path("CO2 Emissions_Canada.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH.resolve()}\n"
        "Download the dataset and place the CSV beside this notebook."
    )


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Import Data
file_path = DATA_PATH
try:
    df = pd.read_csv(file_path)
    print(f"Successfully loaded data from {file_path}\n")
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please check the path and try again.")
    df = None

if df is not None:
    # 2. Primary Analysis
    print("### Primary Analysis ###\n")
    print("#### First 5 rows ####")
    print(df.head())
    print("\n#### Data Info ####")
    df.info()
    print("\n#### Descriptive Statistics ####")
    print(df.describe())

    # 3. Make Matrix Correlation
    print("\n### Correlation Matrix ###")
    # Select only numerical columns for correlation calculation
    numerical_df = df.select_dtypes(include=['number'])

    if not numerical_df.empty:
        correlation_matrix = numerical_df.corr()
        print("Correlation Matrix:\n", correlation_matrix)

        # Visualize the correlation matrix
        plt.figure(figsize=(12, 10))
        sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
        plt.title('Correlation Matrix of Numerical Features')
        plt.show()
    else:
        print("No numerical columns found to compute correlation.")


In [ ]:
# XGBOOST MODEL FOR CO2 EMISSION PREDICTION

!pip -q install xgboost

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load dataset
df = pd.read_csv(DATA_PATH)

# 2. Select independent variables and target
features = [
    "Engine Size(L)",
    "Cylinders",
    "Fuel Consumption Comb (L/100 km)"
]

target = "CO2 Emissions(g/km)"

X = df[features]
y = df[target]

# 3. Split dataset (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

# 4. Create XGBoost model
xgb_model = XGBRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=1.0,
    random_state=42,
    objective="reg:squarederror",
    n_jobs=-1
)

# 5. Train model
xgb_model.fit(X_train, y_train)

print("XGBoost training completed!")

# 6. Make predictions
y_pred = xgb_model.predict(X_test)

# 7. Evaluate model
print("\nXGBoost Model Performance")
print("-------------------------")
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f} g/km")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f} g/km")

# 8. Actual vs predicted values
results = pd.DataFrame({
    "Actual CO2": y_test.values,
    "Predicted CO2": y_pred
})

print("\nActual vs Predicted CO2:")
print(results.head(10))

# 9. Actual vs predicted plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.5)

low = min(y_test.min(), y_pred.min())
high = max(y_test.max(), y_pred.max())

plt.plot(
    [low, high], [low, high],
    "r--", label="Perfect prediction"
)

plt.xlabel("Actual CO2 (g/km)")
plt.ylabel("Predicted CO2 (g/km)")
plt.title("XGBoost: Actual vs Predicted CO2")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# 10. Feature importance
importance = pd.Series(
    xgb_model.feature_importances_,
    index=features
).sort_values(ascending=True)

importance.plot(kind="barh", figsize=(8, 5))
plt.xlabel("Feature Importance")
plt.title("XGBoost Feature Importance")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load dataset
df = pd.read_csv(DATA_PATH)

# 2. Select independent variables and target
features = [
    "Engine Size(L)",
    "Cylinders",
    "Fuel Consumption Comb (L/100 km)"
]
target = "CO2 Emissions(g/km)"

X = df[features]
y = df[target]

# 3. Split dataset (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

# 4. Create Random Forest model
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

# 5. Train model
rf_model.fit(X_train, y_train)
print("Random Forest training completed!")

# 6. Make predictions
y_pred = rf_model.predict(X_test)

# 7. Evaluate model
print("\nModel Performance")
print("-------------------------")
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f} g/km")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f} g/km")

# 8. Actual vs predicted values
results = pd.DataFrame({
    "Actual CO2": y_test.values,
    "Predicted CO2": y_pred
})
print("\nActual vs Predicted CO2:")
print(results.head(10))

# 9. Actual vs predicted plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.5)

low = min(y_test.min(), y_pred.min())
high = max(y_test.max(), y_pred.max())

plt.plot([low, high], [low, high], "r--",
         label="Perfect prediction")

plt.xlabel("Actual CO2 (g/km)")
plt.ylabel("Predicted CO2 (g/km)")
plt.title("Random Forest: Actual vs Predicted CO2")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# 10. Feature importance
importance = pd.Series(
    rf_model.feature_importances_,
    index=features
).sort_values(ascending=True)

importance.plot(kind="barh", figsize=(8, 5))
plt.xlabel("Feature Importance")
plt.title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()

In [ ]:
# RANDOM FOREST: CO2 EMISSION PREDICTION

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load dataset
df = pd.read_csv(DATA_PATH)

# 2. Define features and target
X = df.drop(columns=["CO2 Emissions(g/km)"])
y = df["CO2 Emissions(g/km)"]

# 3. Identify categorical and numerical columns
categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

# 4. Encode categorical variables
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         categorical_features),
        ("num", "passthrough", numerical_features)
    ]
)

# 5. Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

# 6. Create Random Forest model
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    ))
])

# 7. Train model
rf_model.fit(X_train, y_train)

print("\nRandom Forest training completed!")

# 8. Predict CO2 emissions
y_pred = rf_model.predict(X_test)

# 9. Evaluate model
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\nRandom Forest Model Performance")
print("--------------------------------")
print(f"R2 Score: {r2:.4f}")
print(f"MAE: {mae:.4f} g/km")
print(f"RMSE: {rmse:.4f} g/km")

# 10. Actual vs Predicted values
results = pd.DataFrame({
    "Actual CO2": y_test.values,
    "Predicted CO2": y_pred
})

print("\nActual vs Predicted CO2:")
print(results.head(10))

# 11. Actual vs Predicted plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.5)

min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())

plt.plot([min_val, max_val], [min_val, max_val],
         "r--", label="Perfect prediction")

plt.xlabel("Actual CO2 Emissions (g/km)")
plt.ylabel("Predicted CO2 Emissions (g/km)")
plt.title("Random Forest: Actual vs Predicted CO2")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# 12. Feature importance
feature_names = rf_model.named_steps[
    "preprocessor"
].get_feature_names_out()

importance = rf_model.named_steps[
    "model"
].feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
}).sort_values(by="Importance", ascending=False)

print("\nTop 15 Important Features:")
print(importance_df.head(15).to_string(index=False))

# 13. Feature importance plot
top_features = importance_df.head(15).sort_values(
    by="Importance"
)

plt.figure(figsize=(10, 7))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()

In [ ]:
# LINEAR REGRESSION MODEL FOR CO2 EMISSION PREDICTION

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load dataset
df = pd.read_csv(DATA_PATH)

# 2. Select independent variables and target
features = [
    "Engine Size(L)",
    "Cylinders",
    "Fuel Consumption Comb (L/100 km)"
]

target = "CO2 Emissions(g/km)"

X = df[features]
y = df[target]

# 3. Split dataset (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

# 4. Create Linear Regression model
lr_model = LinearRegression()

# 5. Train model
lr_model.fit(X_train, y_train)

print("Linear Regression training completed!")

# 6. Make predictions
y_pred = lr_model.predict(X_test)

# 7. Evaluate model
print("\nLinear Regression Model Performance")
print("-----------------------------------")
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f} g/km")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f} g/km")

# 8. Actual vs predicted values
results = pd.DataFrame({
    "Actual CO2": y_test.values,
    "Predicted CO2": y_pred
})

print("\nActual vs Predicted CO2:")
print(results.head(10).round(2))

# 9. Actual vs predicted plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.5)

low = min(y_test.min(), y_pred.min())
high = max(y_test.max(), y_pred.max())

plt.plot(
    [low, high], [low, high],
    "r--", label="Perfect prediction"
)

plt.xlabel("Actual CO2 (g/km)")
plt.ylabel("Predicted CO2 (g/km)")
plt.title("Linear Regression: Actual vs Predicted CO2")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# 10. Regression coefficients
coefficients = pd.DataFrame({
    "Feature": features,
    "Coefficient": lr_model.coef_
})

print("\nRegression Coefficients:")
print(coefficients.round(4).to_string(index=False))

print("\nIntercept:", round(lr_model.intercept_, 4))

# 11. Coefficient plot
coefficients.sort_values(
    by="Coefficient"
).plot(
    x="Feature",
    y="Coefficient",
    kind="barh",
    legend=False,
    figsize=(8, 5)
)

plt.xlabel("Regression Coefficient")
plt.title("Linear Regression: Feature Coefficients")
plt.grid(axis="x")
plt.tight_layout()
plt.show()

## Interpretation and limitations

- Fuel consumption is closely related to tailpipe CO₂ emissions, so strong test performance is plausible; it should not be interpreted as proof of causal relationships.
- The selected-feature models use engine size, cylinder count, and combined fuel consumption. The full-feature Random Forest also uses the other dataset columns after encoding categorical variables.
- Feature importance indicates how the fitted tree model uses predictors; it is not a causal effect.
- Results depend on the train/test split and dataset composition. For a stronger evaluation, consider cross-validation and testing on a separate set of vehicle model years.
